# Nucleic-Acid C1′ RMSF：單一軌跡交接版

這份 Notebook 一次分析 1 個 PSF + 1 個 DCD。

分析流程：

1. 使用指定 G-tetrad core atoms 將所有 frame 對齊至第 0 frame。
2. 計算每個核酸 residue 的 C1′ atom RMSF。
3. 輸出 NPZ、CSV、PNG、PDF 與執行摘要。

RMSF 定義為每顆 C1′ 相對於其時間平均位置的波動。MDTraj 座標單位為 nm，輸出數據同時保存 nm 與 Å。


## 1. 環境準備

需要 Python 3、MDTraj、NumPy 與 Matplotlib。

    pip install mdtraj numpy matplotlib


In [ ]:
# ============================================================
# 2. 載入套件
# ============================================================

from pathlib import Path
import csv

import mdtraj as md
import matplotlib.pyplot as plt
import numpy as np

print(f"MDTraj: {md.__version__}")
print(f"NumPy: {np.__version__}")


## 3. 使用者設定區

一般情況只需修改 PSF_PATH、DCD_PATH、SYSTEM_NAME 與 OUTPUT_DIR。

重要：MDTraj 的 resid 是從 0 開始的 topology index；本 Notebook 不使用 resid selection，而是直接比對 residue.resSeq，避免 G-tetrad residue number 錯一位。


In [ ]:
# ============================================================
# 4. 使用者設定區：一般情況只修改這一區
# ============================================================

PSF_PATH = Path(
    "/dicos_ui_home/chrysaliso/G4/TO_center_ion_1031/"
    "score606/step3_input.psf"
)

DCD_PATH = Path(
    "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/K/"
    "score_606/rep1/G4_TO_center_606.dcd"
)

SYSTEM_NAME = "K_606_rep1"
CURVE_COLOR = "royalblue"

# 實際 residue sequence numbers，不是零起始 topology indices。
CORE_RESSEQ = [2, 3, 4, 8, 9, 10, 14, 15, 16, 20, 21, 22]

# 用核心區域的 P、C1′、C4′ 共同進行 alignment。
ALIGN_ATOM_NAMES = ["P", "C1'", "C4'"]

# 每個核酸 residue 使用 C1′ 計算 RMSF。
RMSF_ATOM_NAME = "C1'"

# STRIDE = 1 代表使用每個 DCD frame。
TRAJECTORY_STRIDE = 1

# 圖片固定 Y 軸上限；設為 None 時自動依數據決定。
Y_MAX_ANGSTROM = 14.0

OUTPUT_DIR = Path("./results") / SYSTEM_NAME

# CHARMM 與常見核酸 residue names。
NUCLEIC_RESNAMES = {
    "A", "G", "C", "T", "U",
    "DA", "DG", "DC", "DT", "DU",
    "ADE", "GUA", "CYT", "THY", "URA",
}

print(f"System: {SYSTEM_NAME}")
print(f"Trajectory stride: {TRAJECTORY_STRIDE}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## 5. 載入、選取與檢查

程式會確認：

- PSF、DCD 存在且可共同載入
- 每個 CORE_RESSEQ 都存在於核酸 residues
- alignment selection 不為空
- 每個核酸 residue 恰好提供一顆 C1′

若某個 terminal residue 沒有 P atom，仍可由 C1′、C4′ 參與 alignment。


In [ ]:
# ============================================================
# 6. 載入軌跡與建立 selection
# ============================================================

if not PSF_PATH.is_file():
    raise FileNotFoundError(
        f"找不到 PSF：{PSF_PATH}\n"
        "請回到使用者設定區修改 PSF_PATH。"
    )

if not DCD_PATH.is_file():
    raise FileNotFoundError(
        f"找不到 DCD：{DCD_PATH}\n"
        "請回到使用者設定區修改 DCD_PATH。"
    )

if not isinstance(TRAJECTORY_STRIDE, int) or TRAJECTORY_STRIDE <= 0:
    raise ValueError("TRAJECTORY_STRIDE 必須是正整數。")

print(f"Loading trajectory: {DCD_PATH}")
traj = md.load(
    str(DCD_PATH),
    top=str(PSF_PATH),
    stride=TRAJECTORY_STRIDE,
)

if traj.n_frames == 0:
    raise RuntimeError("軌跡沒有任何 frame。")

topology = traj.topology


def is_nucleic_residue(residue):
    """依 residue name 判斷是否為核酸 residue。"""
    return residue.name.upper() in NUCLEIC_RESNAMES


nucleic_residues = [
    residue
    for residue in topology.residues
    if is_nucleic_residue(residue)
]

if not nucleic_residues:
    raise ValueError(
        "沒有辨識到核酸 residues。請檢查 NUCLEIC_RESNAMES。"
    )

available_nucleic_resseq = {
    residue.resSeq
    for residue in nucleic_residues
}

missing_core_resseq = sorted(
    set(CORE_RESSEQ).difference(available_nucleic_resseq)
)

if missing_core_resseq:
    raise ValueError(
        "下列 CORE_RESSEQ 不存在於核酸 residues："
        f"{missing_core_resseq}\n"
        f"可用 resSeq：{sorted(available_nucleic_resseq)}"
    )

align_indices = np.asarray([
    atom.index
    for atom in topology.atoms
    if (
        is_nucleic_residue(atom.residue)
        and atom.residue.resSeq in CORE_RESSEQ
        and atom.name in ALIGN_ATOM_NAMES
    )
], dtype=int)

calc_indices = np.asarray([
    atom.index
    for atom in topology.atoms
    if (
        is_nucleic_residue(atom.residue)
        and atom.name == RMSF_ATOM_NAME
    )
], dtype=int)

if len(align_indices) == 0:
    raise ValueError(
        "Core alignment selection 為空。"
        "請檢查 CORE_RESSEQ 與 ALIGN_ATOM_NAMES。"
    )

if len(calc_indices) == 0:
    raise ValueError(
        f"找不到核酸 {RMSF_ATOM_NAME} atoms。"
        "請檢查 topology atom names。"
    )

calc_atoms = [
    topology.atom(int(atom_index))
    for atom_index in calc_indices
]

calc_resseq = [atom.residue.resSeq for atom in calc_atoms]

if len(calc_resseq) != len(set(calc_resseq)):
    raise ValueError(
        "部分核酸 residue 選到超過一顆 C1′。"
        "若系統包含多條核酸鏈，請增加 chain 或 segment 限制。"
    )

print(f"Loaded frames: {traj.n_frames:,}")
print(f"Total atoms: {traj.n_atoms:,}")
print(f"Core alignment atoms: {len(align_indices)}")
print(f"C1' atoms for RMSF: {len(calc_indices)}")
print(f"Available nucleic resSeq: {sorted(available_nucleic_resseq)}")


## 7. Core alignment 與 RMSF 計算

所有 frame 先以核心原子對齊至第 0 frame。對齊後，RMSF 由下式計算：

$$
\mathrm{RMSF}_i =
\sqrt{
\left\langle
\left\|
\mathbf{r}_i(t)-\left\langle\mathbf{r}_i\right\rangle
\right\|^2
\right\rangle
}
$$

這裡不使用配體原子進行 alignment 或 RMSF calculation。


In [ ]:
# ============================================================
# 8. 執行 alignment、計算 RMSF 並儲存
# ============================================================

reference_frame = traj[0]

# 對齊會套用平移與旋轉到所有 atoms，但只以 core atoms 求轉換矩陣。
traj.superpose(
    reference_frame,
    atom_indices=align_indices,
    ref_atom_indices=align_indices,
)

# MDTraj 座標單位為 nm。
selected_xyz_nm = traj.xyz[:, calc_indices, :]
mean_xyz_nm = np.mean(selected_xyz_nm, axis=0)
displacements_nm = selected_xyz_nm - mean_xyz_nm

rmsf_nm = np.sqrt(
    np.mean(
        np.sum(displacements_nm ** 2, axis=2),
        axis=0,
    )
)
rmsf_angstrom = rmsf_nm * 10.0

residue_names = np.asarray([
    atom.residue.name
    for atom in calc_atoms
])

residue_resseq = np.asarray([
    atom.residue.resSeq
    for atom in calc_atoms
], dtype=int)

residue_labels = np.asarray([
    f"{atom.residue.name}{atom.residue.resSeq}"
    for atom in calc_atoms
])

atom_indices = np.asarray([
    atom.index
    for atom in calc_atoms
], dtype=int)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

npz_path = OUTPUT_DIR / "rmsf_c1_data.npz"
np.savez_compressed(
    npz_path,
    system_name=np.asarray(SYSTEM_NAME),
    curve_color=np.asarray(CURVE_COLOR),
    residue_names=residue_names,
    residue_resseq=residue_resseq,
    residue_labels=residue_labels,
    atom_indices=atom_indices,
    rmsf_nm=rmsf_nm,
    rmsf_angstrom=rmsf_angstrom,
    core_resseq=np.asarray(CORE_RESSEQ, dtype=int),
    align_atom_names=np.asarray(ALIGN_ATOM_NAMES),
    rmsf_atom_name=np.asarray(RMSF_ATOM_NAME),
    trajectory_stride=np.asarray(TRAJECTORY_STRIDE),
    loaded_frames=np.asarray(traj.n_frames),
)

csv_path = OUTPUT_DIR / "rmsf_c1_data.csv"
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "atom_index",
        "residue_name",
        "resSeq",
        "residue_label",
        "rmsf_nm",
        "rmsf_angstrom",
    ])

    for values in zip(
        atom_indices,
        residue_names,
        residue_resseq,
        residue_labels,
        rmsf_nm,
        rmsf_angstrom,
    ):
        writer.writerow([
            int(values[0]),
            str(values[1]),
            int(values[2]),
            str(values[3]),
            f"{values[4]:.6f}",
            f"{values[5]:.6f}",
        ])

summary_path = OUTPUT_DIR / "run_summary.txt"
with summary_path.open("w", encoding="utf-8") as summary_file:
    summary_file.write(f"System: {SYSTEM_NAME}\n")
    summary_file.write(f"PSF: {PSF_PATH}\n")
    summary_file.write(f"DCD: {DCD_PATH}\n")
    summary_file.write(f"Loaded frames: {traj.n_frames}\n")
    summary_file.write(f"Trajectory stride: {TRAJECTORY_STRIDE}\n")
    summary_file.write(f"Core resSeq: {CORE_RESSEQ}\n")
    summary_file.write(f"Alignment atoms: {ALIGN_ATOM_NAMES}\n")
    summary_file.write(f"RMSF atom: {RMSF_ATOM_NAME}\n")
    summary_file.write(f"Number of RMSF atoms: {len(calc_indices)}\n")
    summary_file.write("RMSF reference: time-averaged aligned position\n")
    summary_file.write("MDTraj coordinate unit: nm\n")

print(f"NPZ saved: {npz_path.resolve()}")
print(f"CSV saved: {csv_path.resolve()}")
print(f"Summary saved: {summary_path.resolve()}")


## 9. 從 NPZ 載入並重新排序

若只要調整圖片，可從這裡往下執行，不必重新讀取 DCD。

繪圖順序預設為：

1. Guanine
2. Adenine
3. Thymine
4. Cytosine
5. 其他 residue


In [ ]:
# ============================================================
# 10. 載入 NPZ 與建立 residue 排序
# ============================================================

npz_path = OUTPUT_DIR / "rmsf_c1_data.npz"

if not npz_path.is_file():
    raise FileNotFoundError(
        f"找不到 NPZ：{npz_path}\n"
        "請先執行 RMSF 計算 cell。"
    )

data = np.load(npz_path, allow_pickle=False)

required_keys = {
    "system_name",
    "curve_color",
    "residue_names",
    "residue_resseq",
    "residue_labels",
    "rmsf_angstrom",
}

missing_keys = required_keys.difference(data.files)

if missing_keys:
    raise KeyError(f"NPZ 缺少欄位：{sorted(missing_keys)}")

plot_system_name = str(data["system_name"])
plot_color = str(data["curve_color"])
plot_residue_names = data["residue_names"].astype(str)
plot_residue_labels = data["residue_labels"].astype(str)
plot_rmsf_angstrom = data["rmsf_angstrom"]

RESIDUE_GROUPS = [
    ("G", {"G", "DG", "GUA"}, "green", 0.08, "G region"),
    ("A", {"A", "DA", "ADE"}, "blue", 0.05, "A region"),
    ("T", {"T", "DT", "THY"}, "red", 0.05, "T region"),
    ("C", {"C", "DC", "CYT"}, "purple", 0.04, "C region"),
]

group_indices = {}
already_grouped = set()

for group_name, residue_set, _, _, _ in RESIDUE_GROUPS:
    indices = [
        index
        for index, residue_name in enumerate(plot_residue_names)
        if residue_name.upper() in residue_set
    ]
    group_indices[group_name] = indices
    already_grouped.update(indices)

other_indices = [
    index
    for index in range(len(plot_residue_names))
    if index not in already_grouped
]

new_order = []

for group_name, _, _, _, _ in RESIDUE_GROUPS:
    new_order.extend(group_indices[group_name])

new_order.extend(other_indices)
new_order = np.asarray(new_order, dtype=int)

if len(new_order) != len(plot_residue_names):
    raise RuntimeError("Residue reorder 長度不正確。")

display_labels = [
    plot_residue_labels[index]
    .replace("GUA", "G")
    .replace("ADE", "A")
    .replace("THY", "T")
    .replace("CYT", "C")
    for index in new_order
]

ordered_rmsf = plot_rmsf_angstrom[new_order]
x_positions = np.arange(len(new_order))

print(f"Loaded: {npz_path.resolve()}")
print(f"Residues: {len(new_order)}")


## 11. 繪製單一軌跡 RMSF

單一 DCD 沒有 replicate standard deviation，因此只畫一條 RMSF 曲線，不畫 error band 或散點資料。


In [ ]:
# ============================================================
# 12. RMSF 圖
# ============================================================

fig, ax = plt.subplots(figsize=(18, 10), dpi=300)

# 依重新排序後的群組長度加入背景色。
group_start = 0

for group_name, _, color, alpha, group_label in RESIDUE_GROUPS:
    group_length = len(group_indices[group_name])

    if group_length > 0:
        group_end = group_start + group_length
        ax.axvspan(
            group_start - 0.5,
            group_end - 0.5,
            color=color,
            alpha=alpha,
            linewidth=0,
            label=group_label,
        )
        group_start = group_end

ax.plot(
    x_positions,
    ordered_rmsf,
    color=plot_color,
    linewidth=2.8,
    linestyle="-",
    label=plot_system_name,
)

ax.set_xlabel("Residue", fontsize=30)
ax.set_ylabel("RMSF (Å)", fontsize=30)
ax.set_xlim(-0.5, len(new_order) - 0.5)

if Y_MAX_ANGSTROM is None:
    automatic_y_max = max(1.0, float(np.max(ordered_rmsf)) * 1.10)
    ax.set_ylim(0, automatic_y_max)
else:
    ax.set_ylim(0, Y_MAX_ANGSTROM)

ax.set_xticks(x_positions)
ax.set_xticklabels(
    display_labels,
    rotation=60,
    ha="right",
)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=24,
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4,
)

ax.legend(
    fontsize=18,
    loc="upper right",
    frameon=True,
    framealpha=0.9,
    ncol=2,
)

fig.tight_layout()

png_path = OUTPUT_DIR / "figure_rmsf_c1.png"
pdf_path = OUTPUT_DIR / "figure_rmsf_c1.pdf"

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print(f"PNG saved: {png_path.resolve()}")
print(f"PDF saved: {pdf_path.resolve()}")


## 13. 完成後確認

- CORE_RESSEQ 使用實際 residue sequence numbers，而不是零起始 resid。
- Alignment selection 包含預期的 12 個 G residues。
- 每個核酸 residue 只選到一顆 C1′。
- RMSF 數值在 NPZ 與 CSV 中同時保存 nm、Å。
- 單一軌跡圖沒有 marker、SD 或 error band。
- 若 RMSF 異常偏大，先檢查 G4 是否跨越週期邊界而被拆開，再確認 core selection 是否正確。

若要分析下一條 DCD，只需更換 DCD_PATH、SYSTEM_NAME、CURVE_COLOR，再由上往下重新執行。
